In [ ]:
import sys
sys.path.append('..')
from utils.enhanced_transcript_utils import load_transcript, TranscriptLoader, extract_transcript_metadata
import os
import glob
import json
import pandas as pd
from pathlib import Path
from typing import Dict, List, Any, Optional
from collections import defaultdict
import pprint

def find_all_json_gz_files(base_dir: str = "/home/shared_data") -> List[str]:
    """
    Recursively find all json.gz files in the given directory.
    
    Args:
        base_dir: Base directory to search in
        
    Returns:
        List of paths to json.gz files
    """
    pattern = os.path.join(base_dir, "**", "*.json.gz")
    return glob.glob(pattern, recursive=True)

def process_transcript_file(file_path: str) -> Dict[str, Any]:
    """
    Process a single transcript file and extract usage information.
    
    Args:
        file_path: Path to the transcript file
        
    Returns:
        Dictionary containing metadata and usage information
    """
    try:
        # Load the raw data first to get token usage
        data = TranscriptLoader.load_transcript_file(file_path)
        
        # Extract metadata including token usage
        metadata = extract_transcript_metadata(data)
        
        # Add file path and task information
        path = Path(file_path)
        parts = path.parts
        
        # Try to extract task name and model name from the path
        task_name = None
        model_name = None
        
        if 'baseline_trajectories' in parts:
            idx = parts.index('baseline_trajectories')
            if idx + 1 < len(parts):
                task_name = parts[idx + 1]
        
        # Look for model name in path (usually in format like claude-3-5-sonnet-20241022)
        for part in parts:
            if 'claude' in part.lower() or 'gpt' in part.lower():
                model_name = part
                break
        
        # Add additional information
        result = {
            'file_path': file_path,
            'task_name': task_name or metadata.get('task_name'),
            'model_name': model_name,
            'metadata': metadata
        }
        
        return result
    
    except Exception as e:
        print(f"Error processing {file_path}: {str(e)}")
        return {
            'file_path': file_path,
            'error': str(e)
        }

def aggregate_usage_data(results: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Aggregate usage data across all transcripts.
    
    Args:
        results: List of results from processing transcript files
        
    Returns:
        Dictionary containing aggregated usage information and a dataframe with per-transcript data
    """
    # Initialize aggregation structures
    total_stats = {
        'total_files': len(results),
        'successful_files': 0,
        'total_input_tokens': 0,
        'total_output_tokens': 0,
        'total_tokens': 0,
        'total_cost_usd': 0
    }
    
    # Group by task and model
    by_task = defaultdict(lambda: defaultdict(int))
    by_model = defaultdict(lambda: defaultdict(int))
    
    # Create dataframe data
    df_data = []
    
    for result in results:
        if 'error' in result:
            continue
            
        total_stats['successful_files'] += 1
        
        # Get raw token usage data
        raw_data = TranscriptLoader.load_transcript_file(result['file_path'])
        token_usage = raw_data.get('token_usage', [])
        
        if not token_usage:
            continue
            
        # Get token usage summary from metadata
        usage_summary = result.get('metadata', {}).get('token_usage_summary', {})
        if not usage_summary:
            continue
            
        # Update total stats
        total_stats['total_input_tokens'] += usage_summary.get('total_input_tokens', 0)
        total_stats['total_output_tokens'] += usage_summary.get('total_output_tokens', 0)
        total_stats['total_tokens'] += usage_summary.get('total_tokens', 0)
        total_stats['total_cost_usd'] += usage_summary.get('total_cost_usd', 0)
        
        # Update by task
        task_name = result.get('task_name', 'unknown')
        by_task[task_name]['files'] += 1
        by_task[task_name]['input_tokens'] += usage_summary.get('total_input_tokens', 0)
        by_task[task_name]['output_tokens'] += usage_summary.get('total_output_tokens', 0)
        by_task[task_name]['total_tokens'] += usage_summary.get('total_tokens', 0)
        by_task[task_name]['cost_usd'] += usage_summary.get('total_cost_usd', 0)
        
        # Update by model
        model_name = result.get('model_name', 'unknown')
        by_model[model_name]['files'] += 1
        by_model[model_name]['input_tokens'] += usage_summary.get('total_input_tokens', 0)
        by_model[model_name]['output_tokens'] += usage_summary.get('total_output_tokens', 0)
        by_model[model_name]['total_tokens'] += usage_summary.get('total_tokens', 0)
        by_model[model_name]['cost_usd'] += usage_summary.get('total_cost_usd', 0)
        
        # Create row for dataframe
        # Separate task usage (all entries except the last one) from monitoring usage (last entry)
        task_usage = token_usage[:-1] if len(token_usage) > 1 else []
        monitor_usage = token_usage[-1] if token_usage else {}
        
        # Calculate task totals
        task_input_tokens = sum(entry.get('input_tokens', 0) for entry in task_usage)
        task_output_tokens = sum(entry.get('output_tokens', 0) for entry in task_usage)
        task_total_tokens = sum(entry.get('total_tokens', 0) for entry in task_usage)
        task_cost_usd = sum(entry.get('cost_usd', 0) for entry in task_usage)
        
        # Get monitor totals
        monitor_input_tokens = monitor_usage.get('input_tokens', 0)
        monitor_output_tokens = monitor_usage.get('output_tokens', 0)
        monitor_total_tokens = monitor_usage.get('total_tokens', 0)
        monitor_cost_usd = monitor_usage.get('cost_usd', 0)
        
        # Create row
        row = {
            'file_path': result['file_path'],
            'task_name': task_name,
            'model_name': model_name,
            'task_input_tokens': task_input_tokens,
            'task_output_tokens': task_output_tokens,
            'task_total_tokens': task_total_tokens,
            'task_cost_usd': task_cost_usd,
            'monitor_input_tokens': monitor_input_tokens,
            'monitor_output_tokens': monitor_output_tokens,
            'monitor_total_tokens': monitor_total_tokens,
            'monitor_cost_usd': monitor_cost_usd,
            'total_input_tokens': task_input_tokens + monitor_input_tokens,
            'total_output_tokens': task_output_tokens + monitor_output_tokens,
            'total_tokens': task_total_tokens + monitor_total_tokens,
            'total_cost_usd': task_cost_usd + monitor_cost_usd
        }
        
        df_data.append(row)
    
    # Create dataframe
    usage_df = pd.DataFrame(df_data)
    
    return {
        'total_stats': total_stats,
        'by_task': dict(by_task),
        'by_model': dict(by_model),
        'usage_df': usage_df
    }


In [ ]:




# #%%
# """
# Load and explore a single JSON.gz file to understand its structure
# """
# print("Finding all json.gz files in shared_data directory...")
# files = find_all_json_gz_files()
# print(f"Found {len(files)} json.gz files.")

# if len(files) > 0:
#     # Take the first file for exploration
#     sample_file = files[0]
#     print(f"\nExploring file: {sample_file}")
    
#     # Load the raw data
#     print("\nLoading raw data...")
#     raw_data = TranscriptLoader.load_transcript_file(sample_file)
    
#     # Print the top-level keys
#     print("\nTop-level keys in the JSON file:")
#     for key in raw_data.keys():
#         print(f"- {key}")
    
#     # Process the file to extract metadata
#     print("\nProcessing file to extract metadata...")
#     result = process_transcript_file(sample_file)
    
#     # Print the processed result
#     print("\nProcessed result:")
#     pprint.pprint(result, depth=2)
    
#     # If there's token usage information, print it in detail
#     token_usage = result.get('metadata', {}).get('token_usage_summary', {})
#     if token_usage:
#         print("\nToken usage details:")
#         pprint.pprint(token_usage)
    
#     # Print a sample of the transcript messages if available
#     if 'transcript' in raw_data and 'messages' in raw_data['transcript']:
#         messages = raw_data['transcript']['messages']
#         print(f"\nTranscript contains {len(messages)} messages. Here's a sample of the first message:")
#         if messages:
#             pprint.pprint(messages[0], depth=2)
# else:
#     print("No json.gz files found.")

# # %%
# conversation = raw_data['conversation']
# token_usage = raw_data['token_usage']
# monitor_results = raw_data['monitor_results']
# #%%

# conversation

In [ ]:
# Process all transcript files and create the dataframe
print("Finding all json.gz files in shared_data directory...")
files = find_all_json_gz_files()
print(f"Found {len(files)} json.gz files.")

# Process all files
print("\nProcessing all transcript files...")
results = []
for i, file_path in enumerate(files):
    if i % 10 == 0:
        print(i)
    result = process_transcript_file(file_path)
    results.append(result)
print(f"Processed {len(results)} files.")

# Aggregate the data and get the dataframe
print("\nAggregating usage data...")
aggregated_data = aggregate_usage_data(results)
usage_df = aggregated_data['usage_df']

# Display basic information about the dataframe
print(f"\nCreated dataframe with {len(usage_df)} rows.")
print("\nDataframe columns:")
print(usage_df.columns.tolist())

# Display the first few rows
print("\nFirst few rows of the dataframe:")
print(usage_df.head())

In [ ]:
usage_df.monitor_cost_usd.mean()

In [ ]:
# Basic analysis of the dataframe

# Summary statistics
print("\nSummary statistics:")
print(usage_df[['task_total_tokens', 'monitor_total_tokens', 'total_tokens', 
                'task_cost_usd', 'monitor_cost_usd', 'total_cost_usd']].describe())

# Group by task name
print("\nUsage by task:")
task_summary = usage_df.groupby('task_name').agg({
    'task_total_tokens': 'sum',
    'monitor_total_tokens': 'sum',
    'total_tokens': 'sum',
    'task_cost_usd': 'sum',
    'monitor_cost_usd': 'sum',
    'total_cost_usd': 'sum',
    'file_path': 'count'
}).rename(columns={'file_path': 'count'}).sort_values('total_cost_usd', ascending=False)
print(task_summary)

# Group by model name
print("\nUsage by model:")
model_summary = usage_df.groupby('model_name').agg({
    'task_total_tokens': 'sum',
    'monitor_total_tokens': 'sum',
    'total_tokens': 'sum',
    'task_cost_usd': 'sum',
    'monitor_cost_usd': 'sum',
    'total_cost_usd': 'sum',
    'file_path': 'count'
}).rename(columns={'file_path': 'count'}).sort_values('total_cost_usd', ascending=False)
print(model_summary)


In [ ]:
# Calculate the percentage of tokens and cost used by monitoring
usage_df['monitor_token_percentage'] = (usage_df['monitor_total_tokens'] / usage_df['total_tokens'] * 100).fillna(0)
usage_df['monitor_cost_percentage'] = (usage_df['monitor_cost_usd'] / usage_df['total_cost_usd'] * 100).fillna(0)

print("\nMonitoring overhead statistics:")
print(usage_df[['monitor_token_percentage', 'monitor_cost_percentage']].describe())


In [ ]:
# Save the dataframe to a CSV file
output_path = "experiment_usage_analysis.csv"
usage_df.to_csv(output_path, index=False)
print(f"\nSaved dataframe to {output_path}")


In [ ]:
# Visualization examples (uncomment to use)
# import matplotlib.pyplot as plt
# import seaborn as sns

# plt.figure(figsize=(12, 6))
# sns.boxplot(x='task_name', y='total_cost_usd', data=usage_df)
# plt.title('Cost Distribution by Task')
# plt.xticks(rotation=45)
# plt.tight_layout()
# plt.show()

# plt.figure(figsize=(10, 6))
# sns.scatterplot(x='task_total_tokens', y='monitor_total_tokens', hue='task_name', data=usage_df)
# plt.title('Task Tokens vs Monitor Tokens')
# plt.tight_layout()
# plt.show()
